# Token Position Attribution: Position-Centric Analysis

This notebook demonstrates IRTK's `token_position_attribution` module:

1. **Position gradient attribution** – which positions matter most?
2. **Position flow** – how source info flows through layers
3. **Interaction matrix** – pairwise position interactions
4. **Position ablation** – effect of removing each position
5. **Position to logit** – attribute predictions to positions

## Why This Matters

Token-position attribution traces which input positions contribute to the prediction at each layer. Position-specific gradient analysis and flow tracking reveal the model's information routing — which tokens are read from and when their information reaches the output.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax, jax.numpy as jnp, numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_position_attribution import (
    position_gradient_attribution, position_flow_through_layers,
    position_interaction_matrix, position_specific_ablation,
    position_to_logit_attribution,
)

In [ ]:
cfg = HookedTransformerConfig(n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = [jax.random.normal((key := jax.random.split(key)[1]), l.shape) * 0.1
              if isinstance(l, jnp.ndarray) and l.dtype == jnp.float32 else l for l in leaves]
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 1, 2, 3, 4, 5])
def metric_fn(logits): return float(logits[-1, 0])

In [ ]:
grad_attr = position_gradient_attribution(model, tokens, metric_fn)
print(f"Position scores: {grad_attr['position_scores'].round(4)}")
print(f"Most important: position {grad_attr['most_important_position']}")
print(f"Entropy: {grad_attr['attribution_entropy']:.4f}")

In [ ]:
flow = position_flow_through_layers(model, tokens, source_pos=0)
print(f"Flow shape: {flow['attention_flow'].shape}")
print(f"Persistence: {flow['flow_persistence'].round(4)}")
print(f"Decay rate: {flow['decay_rate']:.4f}")
print(f"Max receiver: position {flow['max_receiver']}")

In [ ]:
interact = position_interaction_matrix(model, tokens, metric_fn)
print(f"Interaction matrix shape: {interact['interaction_matrix'].shape}")
i, j, s = interact['strongest_interaction']
print(f"Strongest: positions ({i}, {j}), score = {s:.4f}")
print(f"Individual effects: {interact['individual_effects'].round(4)}")

In [ ]:
logit_attr = position_to_logit_attribution(model, tokens, target_token=0)
print(f"Position attributions: {logit_attr['position_attributions'].round(4)}")
print(f"Most contributing: position {logit_attr['most_contributing']}")
print(f"Normalized: {logit_attr['normalized_attributions'].round(4)}")